# Analysing an instant messaging experiment

This notebook reads the CSV export of an IM experiment (the **Export data** button of the experiment page writes
`exports/csv/<table>.csv` in the experiment folder; the columns are in the data dictionary of YInstantMessaging)
and reproduces three views of the run: messages per round, who talks to whom, and the share of each kind of action.
Set `EXPORT` to your folder. Pandas and matplotlib are the only requirements.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

EXPORT = Path("exports/csv")  # <experiment folder>/exports/csv
messages = pd.read_csv(EXPORT / "messages.csv")
accounts = pd.read_csv(EXPORT / "accounts.csv")
reactions = pd.read_csv(EXPORT / "reactions.csv")
live = messages[messages["deleted_round"].isna()]
len(live), live["round_id"].max()

## Messages per round

In [ ]:
per_round = live.groupby("round_id").size()
per_round.plot(figsize=(9, 3), xlabel="round", ylabel="messages")
plt.show()

## Kinds of action

The same classification `yim-analyze activity` uses: a forward, a reply or a plain post; reactions are counted apart.

In [ ]:
kind = live.apply(
    lambda m: "forward" if pd.notna(m["fwd_from_message_id"]) else "reply" if pd.notna(m["reply_to_message_id"]) else "post",
    axis=1,
)
shares = kind.value_counts().to_dict()
shares["reaction"] = len(reactions)
pd.Series(shares).sort_values().plot.barh(figsize=(5, 2.5))
plt.show()

## Who replies to whom

In [ ]:
parents = live.set_index("id")["sender_account_id"]
replies = live[live["reply_to_message_id"].notna()].copy()
replies["to"] = replies["reply_to_message_id"].map(parents)
edges = (replies.dropna(subset=["sender_account_id", "to"])
         .groupby(["sender_account_id", "to"]).size().rename("replies").reset_index())
names = accounts.set_index("id")["username"]
edges["from_name"] = edges["sender_account_id"].map(names)
edges["to_name"] = edges["to"].map(names)
edges.sort_values("replies", ascending=False).head(15)

The reports of the experiment page (`Generate report`) go further: cascades, network, homogeneity, emergence. Their JSON files sit in `reports/` and can be loaded with `json.load`.